# A3 ABO · 02 Train

Dùng chung manifest, hai cache, cấu hình head và mã nguồn. Người điều phối chọn chế độ **lock** một lần, các thành viên chọn **batch** để chạy phần được giao, sau đó người điều phối chọn **collect** rồi mở Notebook 03. Chế độ **team** chạy cả suite trên một máy.

Trong Colab, đặt bản ZIP mới ở thư mục chung A3_ABO và bảo đảm mọi người trỏ WORK tới cùng thư mục, hoặc sao chép đúng bộ cohort, ZIP và file lock sang máy của mình.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

ON_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Edit paths here if using another Drive folder or a checkout outside the current directory.
WORK = Path(os.environ.get("A3_WORK", "/content/drive/MyDrive/A3_ABO" if ON_COLAB else "runs/a3/team")).resolve()
REPO = Path(os.environ.get("A3_REPO", ".")).resolve()
if not (REPO / "src/dlbench/a3/curation.py").is_file():
    if not ON_COLAB and (REPO.parent / "src/dlbench/a3/curation.py").is_file():
        REPO = REPO.parent
    elif ON_COLAB:
        bundle = WORK / "A3_ABO_team.zip"
        if bundle.is_file():
            digest = hashlib.sha256(bundle.read_bytes()).hexdigest()[:16]
            REPO = Path("/content") / f"a3-source-{digest}"
            if not (REPO / "src/dlbench/a3/curation.py").is_file():
                shutil.unpack_archive(bundle, REPO)
        elif (WORK / "setup.py").is_file() and (WORK / "src/dlbench/a3/team_batches.py").is_file():
            REPO = WORK
        else:
            raise FileNotFoundError(f"Đặt A3_ABO_team.zip trong WORK hoặc chỉnh WORK. Đang dùng: {WORK}")
    else:
        raise FileNotFoundError("Set A3_REPO to the A3 checkout or extracted source ZIP.")

if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[a3]"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)
env = {**os.environ, "PYTHONPATH": source + os.pathsep + os.environ.get("PYTHONPATH", "")}

def run_a3(*args):
    subprocess.run([sys.executable, "-m", "dlbench.a3.cli", *map(str, args)],
                   cwd=REPO, env=env, check=True)

WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"
IMAGE_ROOT = DATA / "images"
FROZEN = WORK / "cohorts" / "abo-v1"
IMAGE_CACHE = FROZEN / "image.npz"
TITLE_CACHE = FROZEN / "title.npz"
MANIFEST = FROZEN / "manifest.csv"
RUN_ID = os.environ.get("A3_RUN_ID", "abo-v1-main")
SUITE = WORK / "runs" / RUN_ID
INPUT_ARGS = ("--manifest", MANIFEST, "--image-cache", IMAGE_CACHE, "--title-cache", TITLE_CACHE)
print(f"Source: {REPO}\nWorkspace: {WORK}")

def require_cohort(*paths):
    missing = [path.name for path in paths if not path.is_file()]
    if not missing:
        return
    decisions = WORK / "reviews" / "abo-v1" / "decisions.csv"
    if "freeze.json" in missing or "manifest.csv" in missing:
        if FROZEN.exists():
            action = "Thư mục cohort đang thiếu file. Chép lại nguyên cohort đã chốt hoặc chỉnh WORK."
        elif decisions.is_file():
            action = (f"Audit đã có tại {decisions}. Duyệt và lưu file, rồi điền REVIEWER và "
                      "chạy ô chốt manifest trong Notebook 01 Prepare.")
        else:
            action = "Chạy Notebook 01 Prepare để chốt dữ liệu, hoặc chỉnh WORK tới workspace đã chuẩn bị."
    else:
        action = "Chạy ô tạo cache trong Notebook 01 Prepare với ENCODE = True."
    raise RuntimeError(f"Thiếu {', '.join(missing)} ở {FROZEN}. {action}")

## Chọn phần việc

- **lock** tạo khóa benchmark từ cohort và cấu hình hiện tại. Người điều phối thực hiện một lần khi đã có hai cache.
- **batch** chạy mọi kết hợp trong MODELS × SEEDS. Điền PAIRS để chọn cặp chính xác, ví dụ một seed cho model này và hai seed cho model khác. Đặt BATCH_ID riêng cho mỗi batch, ví dụ thien-01.
- **collect** nhận các thư mục batch, báo cặp thiếu/trùng và tạo suite 15 lượt. Mở Notebook 03 sau khi kiểm validation.
- **team** giữ cách chạy cả suite trên một máy. TEAM_SEEDS có thể thu hẹp cho lượt thử.

In [ ]:
MODE = "batch"  # lock | batch | collect | team
BATCH_ID = ""    # Điền ID riêng, ví dụ "thien-01"
MODELS = ["image-only"]
SEEDS = [67, 69]
PAIRS = []       # Ví dụ [("image-only", 67), ("concat", 69)]
BATCH_IDS = []   # Ví dụ ["thien-01", "khoa-01", "khai-01"] khi collect
TEAM_SEEDS = [69420, 67, 69]

In [ ]:
from dlbench.a3.curation import verified_freeze
from dlbench.a3.features import load_cache, inspect_title_policy

TRAIN_DEVICE = os.environ.get("A3_TRAIN_DEVICE", "auto")
CONFIG = REPO / "configs/a3/head.py"
require_cohort(FROZEN / "freeze.json", MANIFEST, IMAGE_CACHE, TITLE_CACHE)
rows = verified_freeze(FROZEN)
receipt = json.loads((FROZEN / "freeze.json").read_text(encoding="utf-8"))
image_meta = load_cache(IMAGE_CACHE, rows, "image")[1]["encoder"]
title_meta = load_cache(TITLE_CACHE, rows, "title")[1]["encoder"]
phrases_path = FROZEN / "extra_mask_phrases.txt"
phrases = phrases_path.read_text(encoding="utf-8-sig").splitlines() if phrases_path.exists() else ()
if (image_meta.get("image_files_sha256") != receipt["image_files_sha256"]
        or title_meta.get("mask_policy") != inspect_title_policy(rows, phrases)):
    raise ValueError("Cache khác dữ liệu hoặc chính sách title đã chốt. Xem notebook 01.")

LOCK = FROZEN / "benchmark_lock.json"
BATCH_ROOT = WORK / "batches"
LOCK_ARGS = (*INPUT_ARGS, "--config", CONFIG, "--lock", LOCK)
if MODE == "lock":
    run_a3("lock-team", *LOCK_ARGS)
    print("Khóa benchmark:", LOCK)
elif MODE == "batch":
    if not BATCH_ID or Path(BATCH_ID).name != BATCH_ID or BATCH_ID in {".", ".."}:
        raise ValueError("Điền BATCH_ID riêng trước khi chạy.")
    jobs = PAIRS or [(model, seed) for model in MODELS for seed in SEEDS]
    if not jobs:
        raise ValueError("Chọn ít nhất một cặp model–seed.")
    batch = BATCH_ROOT / BATCH_ID
    print("Các lượt đã chọn:", jobs)
    run_a3("train-batch", *LOCK_ARGS, "--pairs", *[f"{model}:{seed}" for model, seed in jobs],
           "--output", batch, "--device", TRAIN_DEVICE)
    print("Thư mục bàn giao:", batch)
elif MODE in {"collect", "team"}:
    if MODE == "collect":
        if not BATCH_IDS or any(Path(name).name != name or name in {"", ".", ".."} for name in BATCH_IDS):
            raise ValueError("Điền danh sách ID batch cần gộp.")
        run_a3("collect-team", *LOCK_ARGS, "--batches", *[BATCH_ROOT / name for name in BATCH_IDS],
               "--output", SUITE)
        shutil.copyfile(FROZEN / "freeze.json", SUITE / "data_freeze.json")
        shutil.copyfile(CONFIG, SUITE / "head_config.py")
    else:
        if not (SUITE / "suite.json").is_file():
            run_a3("train-team", *INPUT_ARGS, "--config", CONFIG, "--seeds", *TEAM_SEEDS,
                   "--output", SUITE, "--device", TRAIN_DEVICE)
            shutil.copyfile(FROZEN / "freeze.json", SUITE / "data_freeze.json")
            shutil.copyfile(CONFIG, SUITE / "head_config.py")
        plan = json.loads((SUITE / "suite.json").read_text(encoding="utf-8"))
        if plan.get("profile") != "team" or plan["seeds"] != list(TEAM_SEEDS):
            raise ValueError("Suite hiện có dùng profile hoặc seed khác. Kiểm tra A3_RUN_ID.")
    plan = json.loads((SUITE / "suite.json").read_text(encoding="utf-8"))
    for run in plan["runs"]:
        print(run["model"], run["seed"], run["validation"])
    print("Baseline chốt bằng validation:", plan["unimodal_reference"])
    print("Suite:", SUITE)
else:
    raise ValueError("Chọn MODE là lock, batch, collect hoặc team.")

Nếu batch bị ngắt, chạy lại với cùng BATCH_ID và cùng danh sách cặp. Các lượt đã hoàn thành được kiểm rồi bỏ qua, lượt đang dở được chạy lại. Khi dùng máy riêng, chép nguyên thư mục batch về WORK/batches của người điều phối. Giữ lại cùng A3_RUN_ID ở Notebook 03 và 04.